In [ ]:
import os
import datasets
import pandas as pd
import tqdm
import random

dataset_root = '/data/lihaochen/datasets/TikTok_dataset/'

split = range(1, 341)
# random split range into train and test, with 90% for train and 10% for test
train_split = random.sample(split, int(len(split) * 0.9))
test_split = [i for i in split if i not in train_split]
split = {'train': train_split, 'test': test_split}

for s in ['train', 'test']:
    dataset = []
    for j in sorted(split[s]):
        video_dir = os.path.join(dataset_root, '%05d' % j)
        image_dir = os.path.join(video_dir, 'images')
        video_len = len(os.listdir(image_dir))
        caption_file = open(os.path.join(video_dir, 'caption.txt')).read().strip()
        captions = caption_file.split('\n')
        frames = []

        # generate each frame
        for i in range(1, video_len+1):
            person = f'{j}'
            phi = 0
            mask = os.path.join(video_dir, 'masks', f'{i:04d}.png')
            source = os.path.join(video_dir, 'relight', f'{i:04d}.png')
            target = os.path.join(video_dir, 'images', f'{i:04d}.png')
            img_depth = os.path.join(video_dir, 'img_depth', f'{i:04d}.npy')
            bg_depth = os.path.join(video_dir, 'bg_depth.npy')
            lighting = os.path.join(video_dir, 'hdr', 'refined.exr')
            bg = os.path.join(video_dir, 'inpainted', f'{i:04d}.png')
            caption = captions[i-1]
            if not os.path.exists(img_depth):
                continue
            if not os.path.exists(source):
                continue
            frames.append([person, phi, mask, source, target, img_depth, bg_depth, caption, lighting, bg])

        # group frames into sequences of 4, overlapping by 2
        for i in range(0, len(frames), 2):
            if i + 4 > len(frames):
                dataset.append(frames[-4:])
                break
            dataset.append(frames[i:i+4])

    # gather dataset into a dict
    # dict_dataset = {
    #     'person': [[r[0] for r in frames] for frames in dataset],
    #     'phi': [[r[1] for r in frames] for frames in dataset],
    #     'mask': [[r[2] for r in frames] for frames in dataset],
    #     'source': [[r[3] for r in frames] for frames in dataset],
    #     'target': [[r[4] for r in frames] for frames in dataset],
    #     'img_depth': [[r[5] for r in frames] for frames in dataset],
    #     'bg_depth': [[r[6] for r in frames] for frames in dataset],
    #     'caption': [[r[7] for r in frames] for frames in dataset],
    #     'lighting': [[r[8] for r in frames] for frames in dataset],
    #     'bg': [[r[9] for r in frames] for frames in dataset],
    # }

    # ds = datasets.Dataset.from_dict(dict_dataset)

    # ds.to_parquet(f'./video_{s}.parquet')

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 377.55ba/s]
